In [ ]:
%matplotlib inline

# Get the resampling result

## Problem

Cross-validation, leave-one-out and bootstrap create several ML models
in addition to the one trained from the full training dataset.
By default,
the methods
[compute_cross_validation_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_cross_validation_measure],
[compute_leave_one_out_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_leave_one_out_measure] and
[compute_bootstrap_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_bootstrap_measure]
do not save these models.
How can I get this resampling result?

## Solution

Set the argument `store_resampling_result` of these methods to `True`.

## Step-by-step guide


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
from numpy import array
from numpy import linspace
from numpy import newaxis
from numpy import sin

from gemseo.datasets.io_dataset import IODataset
from gemseo.machine_learning.regression.models.polyreg import PolynomialRegressor
from gemseo.machine_learning.regression.models.polyreg_settings import (
    PolynomialRegressor_Settings,
)
from gemseo.machine_learning.regression.quality.r2_measure import R2Measure

### 1. Define the reference model



In [ ]:
def f(x):
    return (6 * x - 2) ** 2 * sin(12 * x - 4)

### 2. Create the training dataset



In [ ]:
x_train = array([0.1, 0.3, 0.5, 0.6, 0.8, 0.9, 0.95])
y_train = f(x_train)

training_dataset = IODataset()
training_dataset.add_input_group(x_train[:, newaxis], ["x"])
training_dataset.add_output_group(y_train[:, newaxis], ["y"])

### 2. Create the ML model



In [ ]:
regressor = PolynomialRegressor(
    training_dataset, settings=PolynomialRegressor_Settings(degree=3)
)
regressor.learn()

### 3. Evaluate its quality by resampling



In [ ]:
r2 = R2Measure(regressor)

By default, there is no resampling result, especially sub-models



In [ ]:
r2.compute_cross_validation_measure()
assert not regressor.resampling_results

Set the `store_resampling_result` argument to `True` for that purpose



In [ ]:
r2.compute_cross_validation_measure(store_resampling_result=True)
assert regressor.resampling_results

The sub-models can be obtained in this way:



In [ ]:
cross_validation_result = regressor.resampling_results["CrossValidation"]
sub_regressors = cross_validation_result[1]

with `cross_validation_result` of the form `(resampler, ml_models, predictions)`.

Replace `"CrossValidation"` by `"LeaveOneOut"` (resp. `"Bootstrap"`)
in the case of leave-one-out (resp. bootstrap).

## Summary

The sub-models generated by cross-validation can be saved
by setting the argument `store_resampling_result` of
[compute_cross_validation_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_cross_validation_measure],
[compute_leave_one_out_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_leave_one_out_measure] and
[compute_bootstrap_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_bootstrap_measure]
to `True`.

## One step further

In this example,
the submodels can be compared graphically to a test dataset.



In [ ]:
x_test = linspace(0.0, 1.0, 100)
y_test = f(x_test)

plot = plt.plot(x_test, y_test, label="Reference")
plt.plot(x_train, y_train, "o", color=plot[0].get_color(), label="Training dataset")
plt.plot(x_test, regressor.predict(x_test[:, newaxis]), label="Model")
for i, algo in enumerate(sub_regressors):
    plt.plot(x_test, algo.predict(x_test[:, newaxis]), label=f"Sub-model {i}")
plt.legend()
plt.grid()